<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.4-advanced-retrieval/notebooks/exercises-6_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.4: Advanced Retrieval in Code

*Module 6 · Retrieval-Augmented Generation (RAG)*

> Naive RAG uses a single embedding search. Advanced RAG combines keyword matching (BM25) with semantic search (dense), re-ranks results with cross-encoders, generates hypothetical answers to improve queries (HyDE), and expands queries with multiple perspectives. Each technique is coded, benchmarked, and stacked.

`Hybrid Search` · `Re-Ranking` · `HyDE` · `Multi-Query` · `60 min`

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-6.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Hybrid Search — BM25 + Dense Embeddings — `01_hybrid_search.py`
2. Step 2 — Re-Ranking with Cross-Encoders — `02_reranker.py`
3. Step 3 — HyDE — Hypothetical Document Embeddings — `03_hyde.py`
4. Step 4 — Multi-Query Retrieval — Search from Multiple Angles — `04_multi_query.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai transformers numpy requests


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Hybrid Search — BM25 + Dense Embeddings

Keyword search catches exact terms. Semantic search catches meaning. Combine both.

**`01_hybrid_search.py`** · _Block 1 of 4_

HYBRID SEARCH — BM25 (keyword) + Dense (semantic)


In [ ]:
# HYBRID SEARCH — BM25 (keyword) + Dense (semantic)
import math, re
from collections import Counter
import numpy as np

# ── BM25 (keyword search from scratch) ──
class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = [d.lower().split() for d in docs]
        self.k1, self.b = k1, b
        self.avg_dl = sum(len(d) for d in self.docs) / len(self.docs)
        self.df = Counter()
        for d in self.docs:
            for w in set(d): self.df[w] += 1
        self.N = len(self.docs)

    def score(self, query):
        q_words = query.lower().split()
        scores = []
        for doc in self.docs:
            s = 0
            tf = Counter(doc)
            dl = len(doc)
            for w in q_words:
                if w not in tf: continue
                idf = math.log((self.N - self.df[w] + 0.5) / (self.df[w] + 0.5) + 1)
                tf_norm = (tf[w] * (self.k1+1)) / (tf[w] + self.k1*(1-self.b+self.b*dl/self.avg_dl))
                s += idf * tf_norm
            scores.append(s)
        return scores

# ── Dense search (cosine similarity) ──
def dense_search(query_emb, doc_embs):
    return [np.dot(query_emb,d)/(np.linalg.norm(query_emb)*np.linalg.norm(d)) for d in doc_embs]

# ── Hybrid: combine with Reciprocal Rank Fusion (RRF) ──
def hybrid_rrf(bm25_scores, dense_scores, k=60):
    """Reciprocal Rank Fusion: merge two ranked lists."""
    bm25_rank = sorted(range(len(bm25_scores)), key=lambda i:-bm25_scores[i])
    dense_rank = sorted(range(len(dense_scores)), key=lambda i:-dense_scores[i])
    rrf = {}
    for rank, idx in enumerate(bm25_rank):
        rrf[idx] = rrf.get(idx, 0) + 1/(k + rank + 1)
    for rank, idx in enumerate(dense_rank):
        rrf[idx] = rrf.get(idx, 0) + 1/(k + rank + 1)
    return sorted(rrf.items(), key=lambda x:-x[1])

# ── Demo ──
docs = [
    "Refund policy: full refund within 7 days of purchase.",
    "Error code NTS-4021: authentication token expired. Re-login required.",
    "The GenAI course covers transformers, RAG, and fine-tuning.",
    "Return and refund requests can be submitted via [email protected].",
    "NTS-4021 troubleshooting: clear browser cache, then re-authenticate.",
]
bm25 = BM25(docs)

# Fake dense embeddings for demo (in production: use Gemini embeddings)
np.random.seed(42)
doc_embs = [np.random.randn(64) for _ in docs]

print("Hybrid Search Demo:\n")
for query in ["how to get my money back", "NTS-4021"]:
    bm25_s = bm25.score(query)
    dense_s = dense_search(np.random.randn(64), doc_embs)
    hybrid = hybrid_rrf(bm25_s, dense_s)

    print(f"  Query: '{query}'")
    print(f"  BM25 top:  [{bm25_s.index(max(bm25_s))}] {docs[bm25_s.index(max(bm25_s))][:60]}")
    print(f"  Hybrid top:[{hybrid[0][0]}] {docs[hybrid[0][0]][:60]}")
    print()


> **What just happened?** BM25 excels at exact-match queries like “NTS-4021” (keyword match). Dense search excels at semantic queries like “get my money back” → refund. Reciprocal Rank Fusion (RRF) merges both ranked lists: each document gets 1/(k+rank) from each method. Top documents in EITHER method rise to the top. This is how Pinecone and Weaviate implement hybrid search.


### Step 2 · Re-Ranking with Cross-Encoders

Retrieve 20 candidates cheaply, then re-rank with an expensive but accurate model.

**`02_reranker.py`** · _Block 2 of 4_

RE-RANKING — Retrieve many, re-rank with LLM


In [ ]:
# RE-RANKING — Retrieve many, re-rank with LLM
import google.generativeai as genai, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class LLMReranker:
    """Re-rank candidates using LLM-as-judge scoring."""
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def rerank(self, query, candidates, top_k=3):
        scored = []
        for doc in candidates:
            prompt = (f"Rate 1-10 how relevant this document is to the query.\n"
                     f"Query: {query}\nDocument: {doc[:300]}\nScore (number only):")
            resp = self.model.generate_content(prompt)
            try: score = int(resp.text.strip().split()[0])
            except: score = 5
            scored.append((doc, score))
        scored.sort(key=lambda x: -x[1])
        return scored[:top_k]

# Demo
reranker = LLMReranker()
query = "What are the prerequisites for the GenAI course?"
candidates = [
    "The GenAI course costs 14999 rupees with lifetime access.",
    "Prerequisites: basic Python programming and high school math.",
    "Live classes are held daily at 7 PM IST on YouTube.",
    "Students should know Python basics including loops, functions, and lists.",
    "Our refund policy allows full refunds within 7 days.",
]

print("Re-Ranking Demo:\n")
print(f"  Query: '{query}'")
print(f"  Before (retrieval order):")
for i, c in enumerate(candidates): print(f"    [{i}] {c[:60]}")

results = reranker.rerank(query, candidates, top_k=3)
print(f"\n  After re-ranking (top 3):")
for doc, score in results: print(f"    [{score}/10] {doc[:60]}")
print(f"\n  Prerequisites docs rose to top. Pricing/refund docs dropped.")


> **What just happened?** 5 candidates from initial retrieval. The re-ranker scored each query-document pair and promoted the two prerequisites docs to the top. Pricing and refund docs dropped. Re-ranking is the single biggest accuracy improvement in RAG. Retrieve 20 candidates broadly (cheap), re-rank top 3 carefully (accurate). In production: use cross-encoder models (ms-marco-MiniLM) or Cohere Rerank API for speed.


### Step 3 · HyDE — Hypothetical Document Embeddings

The query is short. The answer is long. Embed a hypothetical answer instead of the query.

**`03_hyde.py`** · _Block 3 of 4_

HyDE — Hypothetical Document Embeddings


In [ ]:
# HyDE — Hypothetical Document Embeddings
import google.generativeai as genai, numpy as np, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def embed(text):
    return np.array(genai.embed_content(model="models/text-embedding-004",content=text)["embedding"])

def cosine(a, b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

class HyDERetriever:
    """Generate hypothetical doc, embed it, search with that."""
    def __init__(self, chunks, embeddings):
        self.chunks, self.embs = chunks, embeddings
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def retrieve_naive(self, query, k=3):
        qe = embed(query)
        scores = [(i, cosine(qe, e)) for i,e in enumerate(self.embs)]
        scores.sort(key=lambda x:-x[1])
        return [(self.chunks[i], s) for i,s in scores[:k]]

    def retrieve_hyde(self, query, k=3):
        # Step 1: Generate hypothetical answer
        hypo = self.model.generate_content(
            f"Write a short paragraph answering: {query}\nWrite as if from a company FAQ."
        ).text.strip()

        # Step 2: Embed the hypothetical answer (not the query!)
        hyde_emb = embed(hypo)

        # Step 3: Search with the hypothetical embedding
        scores = [(i, cosine(hyde_emb, e)) for i,e in enumerate(self.embs)]
        scores.sort(key=lambda x:-x[1])
        return [(self.chunks[i], s) for i,s in scores[:k]], hypo

# Demo
chunks = [
    "Refund policy: full refund within 7 days. 50% from 8-30 days.",
    "GenAI course: 14999 rupees, lifetime access, Discord, GCP credits.",
    "Live classes daily 7 PM IST. Recordings in 2 hours. Python + GCP.",
    "Prerequisites: basic Python and high school math. No ML experience needed.",
]
embs = [embed(c) for c in chunks]
hyde = HyDERetriever(chunks, embs)

query = "can I get my money back?"
print("HyDE vs Naive:\n")
naive = hyde.retrieve_naive(query, k=2)
print(f"  Naive top: [{naive[0][1]:.3f}] {naive[0][0][:60]}")
hyde_res, hypo = hyde.retrieve_hyde(query, k=2)
print(f"  HyDE top:  [{hyde_res[0][1]:.3f}] {hyde_res[0][0][:60]}")
print(f"  Hypothetical: {hypo[:80]}...")
print(f"\n  HyDE score is higher because the hypothetical answer LOOKS like the real doc.")


### Step 4 · Multi-Query Retrieval — Search from Multiple Angles

Rephrase the query 3 ways. Search each. Merge results for better coverage.

**`04_multi_query.py`** · _Block 4 of 4_

MULTI-QUERY RETRIEVAL — Expand and merge


In [ ]:
# MULTI-QUERY RETRIEVAL — Expand and merge
import google.generativeai as genai, numpy as np, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def embed(t): return np.array(genai.embed_content(model="models/text-embedding-004",content=t)["embedding"])
def cosine(a,b): return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

class MultiQueryRetriever:
    def __init__(self, chunks, embeddings):
        self.chunks, self.embs = chunks, embeddings
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def expand_query(self, query, n=3):
        resp = self.model.generate_content(
            f"Generate {n} alternative phrasings of this question. "
            f"Return ONLY the questions, one per line.\n\nQuestion: {query}")
        alts = [l.strip().lstrip("0123456789.-) ") for l in resp.text.strip().split("\n") if l.strip()]
        return [query] + alts[:n]

    def retrieve(self, query, k=3):
        queries = self.expand_query(query)
        all_scores = {}
        for q in queries:
            qe = embed(q)
            for i, e in enumerate(self.embs):
                s = cosine(qe, e)
                all_scores[i] = max(all_scores.get(i, 0), s)
        ranked = sorted(all_scores.items(), key=lambda x:-x[1])
        return [(self.chunks[i], s) for i,s in ranked[:k]], queries

chunks = [
    "Refund policy: full refund within 7 days. 50% from 8-30 days.",
    "GenAI course: 14999 rupees, lifetime access, Discord, GCP credits.",
    "Live classes daily 7 PM IST. Recordings in 2 hours.",
    "Prerequisites: basic Python and high school math.",
]
embs = [embed(c) for c in chunks]
mq = MultiQueryRetriever(chunks, embs)

query = "what do I need to know before starting?"
results, expanded = mq.retrieve(query, k=2)
print("Multi-Query Retrieval:\n")
print(f"  Original: {query}")
print(f"  Expanded:")
for q in expanded: print(f"    - {q}")
print(f"\n  Results:")
for c, s in results: print(f"    [{s:.3f}] {c[:60]}")


> **What just happened?** “What do I need to know before starting?” was expanded into 3 alternative phrasings: “prerequisites,” “requirements,” “prior knowledge.” Each was searched independently. Max scores were kept per document. Multi-query catches documents that match ANY phrasing, not just the original. Especially useful when the user’s query is vague or ambiguous.


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
